# Basics of CuPy

In [ ]:
import cupy as cp

x_gpu = cp.array([1,2,3])
print(x_gpu.device)
x_gpu

In [ ]:
y_gpu = cp.array([4,5,6])
sum_xy_gpu = x_gpu + y_gpu
print(sum_xy_gpu.device)
sum_xy_gpu

In [ ]:
z_gpu = cp.dot(x_gpu, y_gpu)
print(z_gpu.device)
z_gpu

In [ ]:
# CuPy to Numpy
array_cpu = cp.asnumpy(x_gpu)
print(type(array_cpu))
array_cpu

In [ ]:
# NumPy to CuPy
array_gpu = cp.asarray(array_cpu)
print(type(array_gpu))
array_gpu

***

# Basics of hipDF

In [ ]:
import cudf

df_gpu = cudf.DataFrame({
    'a':[1,2,3],
    'b':[4,5,6]
})
print(type(df_gpu))
df_gpu

In [ ]:
# Adding a new column
df_gpu['c'] = df_gpu['a'] +  df_gpu['b']
print(type(df_gpu))
df_gpu

In [ ]:
# Filtering base on a condition
filtered_df_gpu = df_gpu[df_gpu['c']>6]
print(type(filtered_df_gpu))
filtered_df_gpu

In [ ]:
import pandas as pd

# hipDF to Pandas
df_pandas = df_gpu.to_pandas()
type(df_pandas)

In [ ]:
# Pandas to hipDF
df_gpu = cudf.DataFrame.from_pandas(df_pandas)
type(df_gpu)

***

# Performance comparison: NumPy vs CuPy

In [ ]:
import numpy as np
import cupy as cp 
from cupyx.profiler import benchmark


def generate_covariance_matrix_numpy(size):

    A_np = np.random.normal(0,1,size = size)
    covariance_matrix_np = np.cov(A_np)

    return covariance_matrix_np


numpy_times = benchmark(generate_covariance_matrix_numpy, ((500,10000),),n_repeat=10).gpu_times
print(f'Processing time cpu: {np.mean(numpy_times)} seconds')


In [ ]:
# CuPy: generating a random matrix of size 500x1000 and computing its covariance

def generate_covariance_matrix_cupy(size):

    A_gpu = cp.random.normal(0,1,size = size)
    covariance_matrix_gpu = cp.cov(A_gpu)

    return covariance_matrix_gpu

cupy_times = benchmark(generate_covariance_matrix_cupy, ((500,10000),),n_repeat=10).gpu_times
print(f'Processing time gpu: {np.mean(cupy_times)} seconds')

***

# Performance comparison: Pandas vs hipDF

In [ ]:
import pandas as pd
import cudf
import numpy as np
import timeit

col_a = list(np.random.rand(1_000_000))
col_b = list(np.random.rand(1_000_000))

def operations_pandas(col_a, col_b):
    df = pd.DataFrame({'A':col_a, 'B':col_b})
    filtered_df = df[df['A']>0.5]
    aggregated_df = filtered_df.groupby('B').agg({'A':'mean'})

pandas_time = timeit.timeit(lambda: operations_pandas(col_a, col_b), globals = globals(), number = 10)

print(f'Processing time cpu: {pandas_time/10} seconds')

In [ ]:
def operations_hipdf(col_a, col_b):
    df = cudf.DataFrame({'A':col_a, 'B':col_b})
    filtered_df = df[df['A']>0.5]
    aggregated_df = filtered_df.groupby('B').agg({'A':'mean'})

hipdf_time = timeit.timeit(lambda: operations_hipdf(col_a, col_b), globals = globals(), number = 10)

print(f'Processing time gpu: {hipdf_time/10} seconds')


***

# Practical example: Implement Markowitz model (Modern Portfolio theory for optimal asset allocation)

In [ ]:
import yfinance as yf
import pandas as pd
import cupy as cp
import cudf
from tqdm import tqdm


def read_tickers(file):
    with open(file, 'r') as f:
        tickers = f.readlines()
        
    tickers = [ticker.strip() for ticker in tickers]

    return tickers

# Text file with the ticker of the stocks we are interested in 
stocks = read_tickers('../data/tickers.txt')



def download_data(start_date, end_date):
    data = {}
    
    for stock in tqdm(stocks):
        ticker = yf.Ticker(stock)
        closing_prices = ticker.history(start=start_date, end=end_date)['Close']
        data[stock] = list(closing_prices.values)

    # hipDF DataFrame from python dictionary    
    data_gpu = cudf.DataFrame(data)

    return data_gpu


# Historical time window of data to acquire. Define START and END dates
start_date = '2015-01-01'
end_date = '2021-01-01'

# Inspect the dataset
dataset = download_data(start_date, end_date)
dataset.head()


## Stocks chart

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()

# Define some colors for each stock line
colors = ['crimson', 'darkgrey', 'skyblue', 'peachpuff', 'thistle']

for stock, color in zip(dataset.columns, colors):
    fig.add_trace(
        go.Scatter(
            # Transform hipDF series to numpy using .to_numpy()
            x = cudf.date_range(start = start_date, end = end_date,freq = 'D').to_numpy(),
            y = dataset[stock].to_numpy(), 
            mode='lines', 
            name = stock, 
            line = dict(color = color)
        )
    )

fig.update_layout(
    title = 'Daily closing prices',
    xaxis_title = '<b>Day</b>',
    yaxis_title = '<b>Price</b>',
    legend_title = 'Stocks', 
    plot_bgcolor = '#f5f5f5',
    paper_bgcolor = '#f0f0f0',
    height = 600
)

fig.update_xaxes(showline = True, linecolor = 'gray', ticks = 'outside', tickcolor='gray', showgrid = True, gridcolor = '#d0d0d0')
fig.update_yaxes(showline = True, linecolor = 'gray', ticks = 'outside', tickcolor='gray', )

fig.show()

### Interoperability between hipDF and cuPy arrays

In [ ]:
# Compute returns on hipDF DataFrame
log_returns = dataset / dataset.shift(1)
log_returns = log_returns.dropna()

In [ ]:
# hipDF DataFrame to cuPy and compute cuPy log of returns 
log_returns = cp.log((log_returns).to_cupy())
log_returns

# Generate and plot the portfolios

In [ ]:
# Number of trading days in a year
NUM_TRADING_DAYS = 252

# The number of random portfolios to generate
NUM_PORTFOLIOS = 10000


def generate_portfolios_cp(returns):
    
    portfolio_means = []
    portfolio_risks = []
    portfolio_weights = []
    portfolio_sharpe_ratio = []

    for _ in range(NUM_PORTFOLIOS):

        w = cp.random.random(len(stocks))
        w /= cp.sum(w)
        
        # Append portfolio normalized weights
        portfolio_weights.append(w)
        
        # Calculate expected returns per asset
        mean_returns = returns.mean(axis = 0)
        
        # Portfolio total return
        portfolio_total_return = cp.sum(cp.asarray(mean_returns)*cp.asarray(w)) * NUM_TRADING_DAYS #252 trading days in a year
        
        # Append portfolio total return
        portfolio_means.append(portfolio_total_return)
        
        # Calculate Covariance of returns
        covariance_returns = cp.cov(returns,rowvar = False)
        
        # cuPy for matrix multiplication
        # Append portfolio total risk(volatility)
        portfolio_total_risk = cp.sqrt(NUM_TRADING_DAYS*cp.dot(w.T, cp.dot(covariance_returns, w)))
        portfolio_risks.append(portfolio_total_risk)
        
        # Append sharpe ratio
        portfolio_sharpe_ratio.append(portfolio_total_return/portfolio_total_risk)
        
        

    return cp.array(portfolio_weights), cp.array(portfolio_means), cp.array(portfolio_risks), cp.array(portfolio_sharpe_ratio)


weights, means, risks, sharpe_ratios  = generate_portfolios_cp(log_returns)



In [ ]:
# Plot portfolios
def plot_portfolios(returns, volatilities, sharpe_ratios):
    fig = go.Figure(
        data = go.Scatter(
            x = volatilities,
            y = returns,
            mode='markers',
            marker = dict(color = sharpe_ratios, showscale = True, colorbar = dict(title = 'Sharpe Ratio'))
        )
    )
    
    
    fig.update_layout(
        title = 'Generated porfolios',
        xaxis_title = '<b>Expected Volatility</b>',
        yaxis_title = '<b>Expected Return</b>',
        plot_bgcolor = '#f5f5f5',
        paper_bgcolor = '#f0f0f0',
        height = 600
    )

    fig.update_xaxes(showline = True, 
                     linecolor = 'gray', 
                     ticks = 'outside', 
                     tickcolor='gray', 
                     showgrid = True, 
                     gridcolor = '#d0d0d0')
    
    fig.update_yaxes(showline = True, 
                     linecolor = 'gray', 
                     ticks = 'outside', 
                     tickcolor='gray')    
    
    
    fig.show()

plot_portfolios(cp.asnumpy(means), cp.asnumpy(risks), cp.asnumpy(sharpe_ratios))

# Get the optimal portfolio using the largest value of the Sharpe Ratio

In [ ]:
# Concatenate output values in a single array using CuPy

results = cp.concatenate([
    means.reshape(-1,1), 
    risks.reshape(-1,1),
    sharpe_ratios.reshape(-1,1),
    weights], axis = 1)

# Transform cuPy array to hipDF DataFrame
results_df = cudf.DataFrame(results, columns = ['Returns', 'Risk', 'Sharpe'] + [stock + "_weight" for stock in stocks])

In [ ]:
# Find portfolio with highest shape ratio
idxmax = results_df['Sharpe'].nlargest(1).index[0]
max_sharpe_portfolio = results_df.iloc[idxmax]
max_sharpe_portfolio

In [ ]:
# Find the portfolio with the minimum risk(std dev)
idxmin = results_df['Risk'].nsmallest(1).index[0]
min_risk_portfolio = results_df.iloc[idxmin]
min_risk_portfolio